In [1]:
import pandas as pd

In [2]:
customers = pd.read_excel("cleaned_data/customers_cleaned.xlsx")

orders = pd.read_excel("cleaned_data/orders_cleaned.xlsx")

order_items = pd.read_excel("cleaned_data/order_items_cleaned.xlsx")

payments = pd.read_excel("cleaned_data/payments_cleaned.xlsx")

products = pd.read_excel("cleaned_data/products_cleaned.xlsx")

reviews = pd.read_excel("cleaned_data/reviews_cleaned.xlsx")

sellers = pd.read_excel("cleaned_data/sellers_cleaned.xlsx")

geolocation = pd.read_excel("cleaned_data/geolocation_cleaned.xlsx")

## check datatypes

In [3]:
print("CUSTOMERS")
print(customers.dtypes)

print("\nORDERS")
print(orders.dtypes)

print("\nORDER ITEMS")
print(order_items.dtypes)

print("\nPAYMENTS")
print(payments.dtypes)

print("\nPRODUCTS")
print(products.dtypes)

print("\nREVIEWS")
print(reviews.dtypes)

print("\nSELLERS")
print(sellers.dtypes)

print("\nGEOLOCATION")
print(geolocation.dtypes)

CUSTOMERS
customer_id                      str
customer_unique_id               str
customer_zip_code              int64
customer_city                    str
customer_state                   str
signup_date           datetime64[us]
gender                           str
age                            int64
dtype: object

ORDERS
order_id                                    str
customer_id                                 str
order_status                                str
order_purchase_timestamp         datetime64[us]
order_approved_at                datetime64[us]
order_delivered_carrier_date     datetime64[us]
order_delivered_customer_date    datetime64[us]
order_estimated_delivery_date    datetime64[us]
dtype: object

ORDER ITEMS
order_id                          str
order_item_id                   int64
product_id                        str
seller_id                         str
shipping_limit_date    datetime64[us]
price                         float64
freight_value                 flo

## feature engineering

#### orders

In [4]:
# Purchase time features, mothly/ quarterly sales analysis.
orders["purchase_month"] = orders["order_purchase_timestamp"].dt.to_period("M")
orders["purchase_quarter"] = orders["order_purchase_timestamp"].dt.quarter
orders["purchase_weekday"] = orders["order_purchase_timestamp"].dt.day_name()

In [5]:
# Delivery performance
orders["delivery_delay"] = (
    orders["order_delivered_customer_date"] -
    orders["order_estimated_delivery_date"]
).dt.days

In [6]:
# How long did the company take to approve an order
orders["processing_time"] = (
    orders["order_approved_at"] -
    orders["order_purchase_timestamp"]
).dt.days

#### order items

In [7]:
# Total order item value
order_items["total_item_value"] = (
    order_items["price"] + order_items["freight_value"]
)

In [8]:
# Estimated profit (business assumption)
order_items["estimated_profit"] = (
    order_items["price"] * 0.2
)

In [9]:
# estimates the profit earned from each order
order_items["estimated_profit"] = (
    order_items["price"] * 0.2
)

#### customers

In [10]:
# Age group segmentation
customers["age_group"] = pd.cut(
    customers["age"],
    bins=[17, 25, 35, 50, 65, 100],
    labels=["18-25", "26-35", "36-50", "51-65", "65+"],
    include_lowest=True
)

#### payments

In [11]:
# Payment behavior. 1 means EMI 0 means Paid once
payments["is_installment"] = payments["payment_installments"].apply(
    lambda x: 1 if x > 1 else 0
)

In [12]:
# is this a high value transaction
payments["high_value_payment"] = payments["payment_value"].apply(
    lambda x: 1 if x > 5000 else 0
)

## aggregate buisness metrics

In [13]:
#### monthly revenue generated from all orders
monthly_sales = order_items.merge(orders, on="order_id")

monthly_sales = monthly_sales.groupby(
    monthly_sales["order_purchase_timestamp"].dt.to_period("M")
)["total_item_value"].sum().reset_index(name="monthly_revenue")

In [14]:
monthly_sales.head()

,order_purchase_timestamp,monthly_revenue
0,2024-05,3.830346e+06
1,2024-06,1.090928e+07
2,2024-07,1.024414e+07
3,2024-08,1.076725e+07
4,2024-09,1.019625e+07


In [15]:
# total revenue generated from each region.
region_revenue = (
    orders.merge(customers, on="customer_id")
    .merge(payments, on="order_id")
)

region_revenue = region_revenue.groupby(
    "customer_state"
)["payment_value"].sum().reset_index(name="state_revenue")

In [16]:
# Product-wise Sales
product_sales = order_items.groupby(
    "product_id"
)["total_item_value"].sum().reset_index(name="product_revenue")

In [17]:
# Payment Method Analysis
payment_analysis = payments.groupby(
    "payment_type"
)["payment_value"].sum().reset_index(name="total_payment")

In [18]:
payment_analysis.head()

,payment_type,total_payment
0,CREDIT_CARD,312290.14
1,DEBIT_CARD,388812.25
2,NET_BANKING,292877.89
3,UPI,390130.56
4,WALLET,274836.48


## analytical Tables Creation

In [19]:
# Customer Summary Table
customer_summary = (
    orders.groupby("customer_id")
    .agg(
        total_orders=("order_id", "count"),
        last_purchase=("order_purchase_timestamp", "max")
    )
    .reset_index()
)

In [20]:
customer_summary.head()

,customer_id,total_orders,last_purchase
0,CUST00002,6,2026-04-28 21:48:15
1,CUST00003,5,2026-01-17 18:51:00
2,CUST00004,2,2026-01-18 12:02:38
3,CUST00005,3,2025-11-27 01:31:09
4,CUST00007,2,2025-09-13 00:19:59


In [21]:
# Product Summary Table
product_summary = (
    order_items.groupby("product_id")
    .agg(
        total_sales=("total_item_value", "sum"),
        total_orders=("order_id", "count")
    )
    .reset_index()
)

In [22]:
product_summary.head()

,product_id,total_sales,total_orders
0,PROD00001,68561.360,24
1,PROD00002,61649.545,19
2,PROD00003,44504.210,20
3,PROD00004,35732.385,14
4,PROD00005,46263.240,16


In [23]:
# Seller Summary Table
seller_summary = (
    order_items.groupby("seller_id")
    .agg(
        total_sales=("total_item_value", "sum"),
        total_orders=("order_id", "count")
    )
    .reset_index()
)

In [24]:
seller_summary.head()

,seller_id,total_sales,total_orders
0,SELL0001,515007.910,183
1,SELL0002,500637.810,183
2,SELL0003,576421.195,197
3,SELL0004,568358.870,203
4,SELL0005,475247.805,180


## Time-Based Performance Metrics

In [25]:
# Daily Sales Trend
daily_sales = order_items.merge(orders, on="order_id")

daily_sales = daily_sales.groupby(
    daily_sales["order_purchase_timestamp"].dt.date
)["total_item_value"].sum().reset_index(name="daily_revenue")

In [26]:
daily_sales.head()

,order_purchase_timestamp,daily_revenue
0,2024-05-21,303178.260
1,2024-05-22,366500.435
2,2024-05-23,250131.720
3,2024-05-24,400945.845
4,2024-05-25,350693.850


In [27]:
# Weekday Purchase Pattern
weekday_sales = orders.groupby(
    "purchase_weekday"
)["order_id"].count().reset_index(name="total_orders")

In [28]:
weekday_sales.head()

,purchase_weekday,total_orders
0,Friday,4276
1,Monday,4376
2,Saturday,4289
3,Sunday,4155
4,Thursday,4375


## convert date columns to mysql format

#### customers

In [47]:
customers["signup_date"] = pd.to_datetime(customers["signup_date"], dayfirst=True).dt.strftime("%Y-%m-%d")

C:\Users\Tanmay\AppData\Local\Temp\ipykernel_12744\949591934.py:1: UserWarning: Parsing dates in %Y-%m-%d format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  customers["signup_date"] = pd.to_datetime(customers["signup_date"], dayfirst=True).dt.strftime("%Y-%m-%d")


#### orders

In [ ]:
orders["order_purchase_timestamp"] = pd.to_datetime(
    orders["order_purchase_timestamp"]
).dt.strftime("%Y-%m-%d %H:%M:%S")

orders["order_approved_at"] = pd.to_datetime(
    orders["order_approved_at"]
).dt.strftime("%Y-%m-%d %H:%M:%S")

orders["order_delivered_carrier_date"] = pd.to_datetime(
    orders["order_delivered_carrier_date"]
).dt.strftime("%Y-%m-%d %H:%M:%S")

orders["order_delivered_customer_date"] = pd.to_datetime(
    orders["order_delivered_customer_date"]
).dt.strftime("%Y-%m-%d %H:%M:%S")

orders["order_estimated_delivery_date"] = pd.to_datetime(
    orders["order_estimated_delivery_date"]
).dt.strftime("%Y-%m-%d %H:%M:%S")

#### order items

In [ ]:
order_items["shipping_limit_date"] = pd.to_datetime(
    order_items["shipping_limit_date"],
    dayfirst=True
).dt.strftime("%Y-%m-%d %H:%M:%S")

#### reviews

In [50]:
reviews["review_creation_date"] = pd.to_datetime(
    reviews["review_creation_date"],
    dayfirst=True
).dt.strftime("%Y-%m-%d %H:%M:%S")

reviews["review_answer_timestamp"] = pd.to_datetime(
    reviews["review_answer_timestamp"],
    dayfirst=True
).dt.strftime("%Y-%m-%d %H:%M:%S")

C:\Users\Tanmay\AppData\Local\Temp\ipykernel_12744\1009860185.py:1: UserWarning: Parsing dates in %Y-%m-%d %H:%M:%S format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  reviews["review_creation_date"] = pd.to_datetime(
C:\Users\Tanmay\AppData\Local\Temp\ipykernel_12744\1009860185.py:6: UserWarning: Parsing dates in %Y-%m-%d %H:%M:%S format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  reviews["review_answer_timestamp"] = pd.to_datetime(


## remove duplicates remaining

In [29]:
order_items = order_items.drop_duplicates(subset=["order_id", "order_item_id"])

In [30]:
order_items.duplicated(subset=["order_id", "order_item_id"]).sum()

np.int64(0)

## save transformed data

In [29]:
import os
os.makedirs("transformed_data", exist_ok=True)

In [30]:
customers.to_excel("transformed_data/customers_transformed.xlsx", index=False)

orders.to_excel("transformed_data/orders_transformed.xlsx", index=False)

order_items.to_excel("transformed_data/order_items_transformed.xlsx", index=False)

payments.to_excel("transformed_data/payments_transformed.xlsx", index=False)

products.to_excel("transformed_data/products_transformed.xlsx", index=False)

reviews.to_excel("transformed_data/reviews_transformed.xlsx", index=False)

sellers.to_excel("transformed_data/sellers_transformed.xlsx", index=False)

geolocation.to_excel("transformed_data/geolocation_transformed.xlsx", index=False)

In [51]:
customers.to_csv("transformed_data_csv/customers_transformed.csv", index=False)

orders.to_csv("transformed_data_csv/orders_transformed.csv", index=False)

order_items.to_csv("transformed_data_csv/order_items_transformed.csv", index=False)

payments.to_csv("transformed_data_csv/payments_transformed.csv", index=False)

products.to_csv("transformed_data_csv/products_transformed.csv", index=False)

reviews.to_csv("transformed_data_csv/reviews_transformed.csv", index=False)

sellers.to_csv("transformed_data_csv/sellers_transformed.csv", index=False)

geolocation.to_csv("transformed_data_csv/geolocation_transformed.csv", index=False)

In [ ]:
# done

In [31]:
order_items.to_csv("transformed_data_csv/order_items_transformedd.csv", index=False)